# Overnight CPU profile comparison

Four population profiles × three learning methods × one shared seed, 16,384 training decisions per condition. No GPU or hosted API is used. This explores profile and inheritance differences on seed 11; it does not measure consistency across independent seeds. Read [the design and results review](../docs/overnight-cpu.md).

Run All initially only displays the plan and saved results. Enable the benchmark or training explicitly below. The same output directory resumes interrupted training and skips completed conditions. Keep the machine awake; the kernel must remain running.

The readiness check has not yet beaten random survival. Inspect learning, feeding and reproduction before interpreting differences between profiles.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
CONFIG = "configs/overnight-cpu.yaml"
OUTPUT = "runs/overnight-profiles"
HOURS = 10
SEEDS = [11]  # Shared across all four profiles and all three methods.
RUN_BENCHMARK = False
RUN_TRAINING = False

def launch(mode):
    cmd = [sys.executable, "-u", "-m", "millstlabs.batch", "--config", CONFIG,
           "--output", OUTPUT, "--mode", mode, "--hours", str(HOURS)]
    if SEEDS is not None:
        cmd += ["--seed", *map(str, SEEDS)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
            if proc.wait():
                raise RuntimeError(f"Runner exited with code {proc.returncode}; inspect output above.")
        except KeyboardInterrupt:
            proc.terminate()
            proc.wait()
            print("Stopped. The next invocation resumes the last saved checkpoint.")
            raise
launch("plan")

## Measure this machine

Set `RUN_BENCHMARK = True` in the settings cell to measure CPU throughput and peak memory (about two minutes on the development Mac). The benchmark stops at extinction or 128 ticks. Its timing excludes warm starts, evaluation, model loading, and inheritance probes. Do not multiply a four-tick smoke run into an overnight estimate.

In [ ]:
if RUN_BENCHMARK:
    launch("benchmark")
else:
    print("Benchmark disabled; set RUN_BENCHMARK=True to measure this machine.")

## Execute or resume

Set `RUN_TRAINING = True` in the settings cell, then rerun it and this cell. The first time includes one 4,096-demonstration warm start shared across all 12 conditions. Runs execute serially with four CPU threads. The ten-hour limit is soft: an in-progress warm start, evaluation, or optimizer update completes before pausing. Rerun with the same config/output to continue.

For an unattended Mac run, the terminal alternative is `caffeinate -i bash scripts/overnight.sh --mode run --hours 10`. From the repository root on either Mac or Codespaces, `bash scripts/overnight.sh --mode run --hours 10` runs the same experiment.

In [ ]:
if RUN_TRAINING:
    launch("run")
else:
    print("Training disabled. Set RUN_TRAINING=True when ready for the overnight run.")

## Review saved results

Restricted mean lifetime is capped at 512 ticks. Each panel is one population profile, with the three methods shown separately. All conditions share seed 11 and the same warm start. Initial-to-final changes use only their common evaluation maps; final comparisons use all four maps. This is a single-seed exploration, not a replicated test. No births means the run has not tested inheritance.

In [ ]:
from millstlabs.batch import plan
from millstlabs.config import load_config

root = ROOT / OUTPUT
if (root / "config.yaml").exists():
    saved = load_config(root / "config.yaml")
    assert saved.digest() == load_config(CONFIG).digest(), "Output belongs to another configuration"
    # Read live logs directly; leave status/analysis generation to the runner after it exits.
    rows = []
    for condition in plan(saved)["runs"]:
        directory = root / f"{condition['profile']}-{condition['method']}-s{condition['seed']}"
        evaluation = directory / "evaluation.jsonl"
        if evaluation.exists():
            rows += [dict(run=directory.name, profile=condition["profile"], method=condition["method"], **json.loads(line)) for line in evaluation.read_text().splitlines()]
    if rows:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(len(saved.profiles), 1, figsize=(9, 3 * len(saved.profiles)), squeeze=False)
        for ax, profile in zip(axes[:, 0], saved.profiles):
            for method in saved.methods:
                points = sorted([r for r in rows if r["profile"] == profile and r["method"] == method], key=lambda r: r["checkpoint"])
                if points:
                    ax.plot([r["checkpoint"] for r in points], [r["restricted_mean_lifetime"] for r in points], marker="o", label=method)
            ax.set(title=profile, xlabel="Training decision checkpoint", ylabel="Mean lifetime (ticks)", ylim=(0, 512))
            if ax.lines:
                ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()
    else:
        print("No evaluation saved yet.")
    status = root / "status.json"
    if status.exists():
        result = json.loads(status.read_text())
        print(f"At last runner exit: {result['complete']}/{result['total']} conditions complete")
        for row in result["runs"]:
            print(row["profile"], row["method"], row["seed"], row["status"], row.get("final_lifetime"), row.get("lifetime_change_on_common_maps"))
else:
    print("No overnight results yet.")